# Amazon ML Challenge 2026: Entity Resolution Suite
### Google Colab GPU/CPU Runner with Google Drive Persistence & Auto-Resume

Run this notebook in **Google Colab** with a **GPU runtime**:
`Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU` (or high-RAM CPU).

This entity resolution pipeline matches companies across noisy sources (Source 1, Source 2, Source 3):
1. **Google Drive Integration**: Mirrored after `minor-project` and `multi-horizon-ofi`. Automatically mounts `/content/drive/MyDrive/amazon-ml-challenge-2026` for persistent storage of datasets (`data/`), model weights (`weights/`), and outputs (`output/`).
2. **Sharded Data Loading**: Uses the 500k-row compressed `.tsv.gz` shards (`data/shards/train` and `data/shards/test`) for fast Drive upload and zero-loss streaming.
3. **Crash Resumption (`--resume`)**: Atomic stage checkpointing. If Colab disconnects or times out, re-running the cell instantly resumes from the exact last completed batch.
4. **Proactive Memory & OOM Protection**: Real-time RAM/VRAM monitoring with garbage collection and dynamic batch throttling.
5. **Zero-Leakage 3-Tier Scorecard**: Evaluates Pair Recall (Blocking), Precision/Recall (Classifier), and Macro F0.5 (Decision Layer) separately.
6. **Official Competition Audit**: Runs `utils/validate_submission.py` to ensure compliance before saving.

In [ ]:
# 1. Verify GPU availability and system resources
!nvidia-smi || echo "Running in CPU Multi-Core Mode"
!lscpu | grep -E "Model name|CPU\(s\):" || true

In [ ]:
# 2. Mount Google Drive to persist all datasets, weights, and benchmark results
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

In [ ]:
# 3. Initialize Google Drive directory structure for amazon-ml-challenge-2026
import os
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/amazon-ml-challenge-2026")
DRIVE_DATA = DRIVE_ROOT / "data"
DRIVE_WEIGHTS = DRIVE_ROOT / "weights"
DRIVE_CHECKPOINTS = DRIVE_ROOT / "checkpoints"
DRIVE_OUTPUT = DRIVE_ROOT / "output"

for folder in [DRIVE_ROOT, DRIVE_DATA, DRIVE_WEIGHTS, DRIVE_CHECKPOINTS, DRIVE_OUTPUT]:
    folder.mkdir(parents=True, exist_ok=True)

print(f"[✓] Google Drive structure initialized at: {DRIVE_ROOT}")
print(f"    - Datasets Dir   : {DRIVE_DATA}")
print(f"    - Weights Dir    : {DRIVE_WEIGHTS}")
print(f"    - Checkpoints Dir: {DRIVE_CHECKPOINTS}")
print(f"    - Output Dir     : {DRIVE_OUTPUT}")

In [ ]:
# 4. Clone or pull the repository
import os
REPO_URL = "https://github.com/Suyash8/amazon-ml-challenge.git"
REPO_DIR = "/content/amazon-ml-challenge"

if not os.path.exists(REPO_DIR):
    !git clone $REPO_URL $REPO_DIR
%cd $REPO_DIR
!git pull || true

In [ ]:
# 5. Install required packages (rapid string matching, lightgbm, etc.)
!python install.py

In [ ]:
# 6. Dataset Linking, Shards Setup & Verification
# Google Drive datasets (/content/drive/MyDrive/amazon-ml-challenge-2026/data/)
# are symlinked to local data/ for zero-copy high-throughput I/O.
import os
from pathlib import Path

!mkdir -p data
drive_data = Path("/content/drive/MyDrive/amazon-ml-challenge-2026/data")
if drive_data.exists():
    for item in drive_data.iterdir():
        target = Path(f"data/{item.name}")
        if not target.exists():
            os.symlink(str(item), str(target))
            print(f"[✓] Linked {item.name} from Google Drive -> data/{item.name}")

# Check available dataset paths
print("\n--- AVAILABLE DATASETS CHECK ---")
for p in ["data/shards/train", "data/shards/test", "dataset_split/train", "dataset_split/test"]:
    if Path(p).exists():
        print(f"  [READY] Found dataset: {p}")
    else:
        print(f"  [INFO] Path not local: {p} (will look in Google Drive if uploaded)")

In [ ]:
# 7. Fast Smoke Test (~10 seconds)
# Verifies normalization, candidate blocking, classifier, decision layer & audit end-to-end
!python scripts/run_pipeline.py \
    --train-dir dataset_split/train \
    --test-dir dataset_split/test \
    --drive-sync-dir /content/drive/MyDrive/amazon-ml-challenge-2026 \
    --smoke-test

In [ ]:
# 8. Zero-Leakage 5-Fold Cross-Validation
# Tracks Blocking Recall, Classifier Precision/Recall, and Decision Layer Macro F0.5
!python src/validation.py \
    --mode cv \
    --train-dir dataset_split/train \
    --n-splits 5

In [ ]:
# 9. Full Pipeline Execution: Training, Blocking & Chunked Inference (Crash-Resilient with Auto-Resume)
# Automatically uses GPU if available, throttles RAM safely, and checkpoints every chunk to Google Drive.
# If interrupted or timed out, re-running this cell resumes from the exact last completed batch!

# Uses sharded dataset if present in data/shards, otherwise falls back to dataset_split
import os
train_path = "data/shards/train" if os.path.exists("data/shards/train") else "dataset_split/train"
test_path = "data/shards/test" if os.path.exists("data/shards/test") else "dataset_split/test"

print(f"Running pipeline with Train: {train_path} | Test: {test_path}")

!python scripts/run_pipeline.py \
    --train-dir $train_path \
    --test-dir $test_path \
    --drive-sync-dir /content/drive/MyDrive/amazon-ml-challenge-2026 \
    --device auto \
    --batch-size 2000 \
    --n-jobs -1

In [ ]:
# 10. Run Official Submission Validator on Generated Files
# Validates format, ID compliance, and required columns for competition submission
import os
test_path = "data/shards/test" if os.path.exists("data/shards/test") else "dataset_split/test"

!python utils/validate_submission.py \
    --matching output/matching_results.tsv \
    --candidate output/candidate_pairs.tsv \
    --test-dir $test_path \
    --check-ids

In [ ]:
# 11. Display Scorecard & Submission Preview Inline
import json
from pathlib import Path
from IPython.display import display, Markdown

scorecard_path = Path("output/scorecard.json")
if scorecard_path.is_file():
    sc = json.loads(scorecard_path.read_text())
    md_table = f"""
### 📊 Evaluation Scorecard (3 Pipeline Tiers)
| Pipeline Tier | Metric | Value |
| :--- | :--- | :--- |
| **1. Candidate Blocking** | Pair Recall | **{sc.get('blocking_pair_recall', 0)*100:.2f}%** |
| | Space Reduction Rate | **{sc.get('candidate_reduction_rate', 0)*100:.2f}%** |
| | Candidates / Entity | **{sc.get('candidates_per_entity', 0):.1f}** |
| **2. GBDT Classifier** | Pair Precision | **{sc.get('classifier_pair_precision', 0)*100:.2f}%** |
| | Pair Recall | **{sc.get('classifier_pair_recall', 0)*100:.2f}%** |
| | Pairwise F1 | **{sc.get('classifier_pair_f1', 0):.4f}** |
| **3. Decision Layer** | **Official Macro F0.5** | **{sc.get('official_macro_f05', 0):.4f}** |
| | Singleton Sub-Score | **{sc.get('singleton_macro_f05', 0):.4f}** |
| | Matched Sub-Score | **{sc.get('matched_macro_f05', 0):.4f}** |
"""
    display(Markdown(md_table))

print("\n--- First 5 rows of candidate_pairs.tsv ---")
!head -n 6 output/candidate_pairs.tsv
print("\n--- First 5 rows of matching_results.tsv ---")
!head -n 6 output/matching_results.tsv